<a href="https://colab.research.google.com/github/claudiorafaels/nondeterminism_llm_inference/blob/main/labs_2026_04_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Célula 1
!pip install vllm sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2

In [1]:
from huggingface_hub import login
from google.colab import userdata

# Puxa o token secreto que você salvou com o nome "HF_TOKEN"
meu_token = userdata.get('HF_TOKEN')

# Faz o login silenciosamente, sem pedir para você digitar nada
login(token=meu_token)

print("Autenticado com sucesso via Secrets!")

Autenticado com sucesso via Secrets!


In [ ]:
# Célula 3: Motor 7B Blindado (Forçando variáveis no Sistema Operacional)
%env VLLM_USE_V1=0
%env VLLM_WORKER_MULTIPROC_METHOD=spawn

from vllm import LLM, SamplingParams

print("Iniciando Qwen 7B na GPU A100 (BFloat16 + Deep Layers)...")

try:
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct",
        dtype="bfloat16",
        max_model_len=1024,          # Reduzido preventivamente para evitar estouro de KV Cache
        gpu_memory_utilization=0.8,  # Margem segura para o S.O.
        enforce_eager=False,         # Permite o CUDA Graphs (necessário para o erro)
        distributed_executor_backend="uni"
    )
    print("\n✅ Motor de 7B carregado! VRAM alocada com sucesso.")
except Exception as e:
    print(f"\n❌ Falha: {e}")

In [5]:
# Célula 4/5: Simulador de Tráfego Dinâmico (Busca pelo Colapso Semântico)
import random
from vllm import SamplingParams

# Exigência do TCC: Determinismo Absoluto (Temp 0, Top-P 1.0, Seed fixa)
sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=10, seed=42)

# O seu prompt "do mundo real"
prompt_bruto = "Defina o conceito de Transformação Digital em exatamente 6 palavras, nem mais, nem menos."
msg = [{"role": "user", "content": prompt_bruto}]
prompt_formatado = llm.get_tokenizer().apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

# 1. Estabelecer o Padrão Ouro (Lote Isolado = Ambiente Controlado)
print("Estabelecendo Referência (Lote de 1)...")
out_ref = llm.generate([prompt_formatado], sampling_params, use_tqdm=False)
texto_referencia = out_ref[0].outputs[0].text.strip()
print(f"Padrão Ouro: '{texto_referencia}'\n")

# 2. Execução de Monte Carlo (Simulando Variação Extrema de Servidor)
num_iteracoes = 100
textos_diferentes_encontrados = set()

print(f"Iniciando {num_iteracoes} execuções com Carga Dinâmica Flutuante e Fragmentação de KV Cache...\n")

for i in range(num_iteracoes):
    # Maximizando o chicote de carga (de 1 até quase o limite do motor)
    carga_atual = random.randint(1, 250)

    # Fragmentação Extrema: PagedAttention lidando com blocos de 10 a 500 palavras
    prompts_ruido = [f"Explain AI in {random.randint(10, 500)} words."] * carga_atual

    lote_total = [prompt_formatado] + [
        llm.get_tokenizer().apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
        for p in prompts_ruido
    ]

    # Executa a geração sob o tráfego fragmentado atual
    out_stress = llm.generate(lote_total, sampling_params, use_tqdm=False)
    texto_stress = out_stress[0].outputs[0].text.strip()

    # Monitora o colapso semântico
    if texto_stress != texto_referencia:
        print(f"🔥 COLAPSO DETECTADO NA ITERAÇÃO {i+1} (Carga: {carga_atual})")
        print(f"   Referência : {texto_referencia}")
        print(f"   Modificado : {texto_stress}")
        textos_diferentes_encontrados.add(texto_stress)

    # Feedback de progresso a cada 10 iterações
    if (i+1) % 10 == 0:
        print(f"Progresso: {i+1}/{num_iteracoes} concluídas...")

# --- VEREDITO ACADÊMICO ---
print("\n" + "="*70)
if len(textos_diferentes_encontrados) > 0:
    print(f"SUCESSO TOTAL! Foram encontradas {len(textos_diferentes_encontrados)} variações semânticas.")
    print("O TCC provou que o Batching Dinâmico quebra o Argmax em ambiente de produção.")
else:
    print("O modelo resistiu às 100 iterações. A invariância de lote do vLLM superou o estresse de Monte Carlo.")
print("="*70)

Estabelecendo Referência (Lote de 1)...
Padrão Ouro: 'Inovação digital para negócios transformados'

Iniciando 100 execuções com Carga Dinâmica Flutuante e Fragmentação de KV Cache...

🔥 COLAPSO DETECTADO NA ITERAÇÃO 1 (Carga: 112)
   Referência : Inovação digital para negócios transformados
   Modificado : Inovação digital para processos corporativos.
🔥 COLAPSO DETECTADO NA ITERAÇÃO 2 (Carga: 42)
   Referência : Inovação digital para negócios transformados
   Modificado : Inovação digital para processos corporativos.
🔥 COLAPSO DETECTADO NA ITERAÇÃO 4 (Carga: 134)
   Referência : Inovação digital para negócios transformados
   Modificado : Inovação digital para processos corporativos.
🔥 COLAPSO DETECTADO NA ITERAÇÃO 5 (Carga: 183)
   Referência : Inovação digital para negócios transformados
   Modificado : Inovação digital para processos corporativos.
🔥 COLAPSO DETECTADO NA ITERAÇÃO 6 (Carga: 165)
   Referência : Inovação digital para negócios transformados
   Modificado : Inovação dig

In [6]:
# Célula 6: Análise de Similaridade Semântica (Veredito Final Monte Carlo)
from sentence_transformers import SentenceTransformer, util

print("Carregando modelo de Embeddings (all-MiniLM-L6-v2)...")
modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')

print("\n" + "="*70)
print("ANÁLISE DE IMPACTO SEMÂNTICO (TESTE DE ESTRESSE MONTE CARLO)")
print("="*70)

# Verifica se o simulador encontrou alguma quebra de determinismo
if not textos_diferentes_encontrados:
    print("\n💎 RESULTADO NEGATIVO (Resiliência Total):")
    print("O modelo manteve 100% de identidade semântica em todas as 100 iterações.")
    print("A infraestrutura absorveu a carga dinâmica extrema sem vazar erros para o texto final.")
else:
    print(f"\n🔥 RESULTADO POSITIVO: {len(textos_diferentes_encontrados)} variações detectadas!")
    print("Calculando o desvio de significado (Drift Semântico)...\n")

    # 1. Gerar o vetor de referência (Padrão Ouro)
    emb_ref = modelo_emb.encode(texto_referencia, convert_to_tensor=True)

    # 2. Analisar cada variação isoladamente
    for i, texto_variante in enumerate(textos_diferentes_encontrados):
        emb_variante = modelo_emb.encode(texto_variante, convert_to_tensor=True)
        similaridade = util.cos_sim(emb_ref, emb_variante).item() * 100

        print(f"--- Variação {i+1} ---")
        print(f"Texto: '{texto_variante}'")
        print(f"Similaridade de Cosseno: {similaridade:.4f}%")
        print(f"Desvio Semântico (Erro): {100 - similaridade:.4f}%\n")

    print("-" * 70)
    print("Impacto para o TCC:")
    print("- A execução sob Carga Dinâmica Flutuante alterou a saída final do modelo.")
    print("- Isso prova que a fragmentação de memória quebra a reprodutibilidade,")
    print("- evidenciando que o hardware influencia na regra de negócios da aplicação.")

Carregando modelo de Embeddings (all-MiniLM-L6-v2)...

ANÁLISE DE IMPACTO SEMÂNTICO (TESTE DE ESTRESSE MONTE CARLO)

🔥 RESULTADO POSITIVO: 1 variações detectadas!
Calculando o desvio de significado (Drift Semântico)...

--- Variação 1 ---
Texto: 'Inovação digital para processos corporativos.'
Similaridade de Cosseno: 76.1476%
Desvio Semântico (Erro): 23.8524%

----------------------------------------------------------------------
Impacto para o TCC:
- A execução sob Carga Dinâmica Flutuante alterou a saída final do modelo.
- Isso prova que a fragmentação de memória quebra a reprodutibilidade,
- evidenciando que o hardware influencia na regra de negócios da aplicação.
